# ModalityRouter

> Notebook generated from [`examples/multimodal/04_modality_router.py`](../../examples/multimodal/04_modality_router.py).

| Field | Value |
|------|-------|
| **Dataset** | Mixed (embedded, 18 labeled messages) |
| **API key** | ✅ **No API keys required** — uses stubs/mocks. |

**Expected result:** Classification accuracy, LangGraph routing and force_modality override demo.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
ModalityRouter — Deterministic modality classification and LangGraph node
============================================================================
Component: SPEC-MM-AGT-004 / prismal.agents.multimodal.modality_router

Dataset: ATIS + arXiv + ActivityNet (mixed) — 18 labeled messages
  • We combine:
      - ATIS-style messages ("Show me flights…") → TEXT
      - image/png attachments → IMAGE
      - audio/wav attachments → AUDIO
      - video/mp4 attachments → VIDEO
      - Combinations (text + image + audio) → MIXED
      - Intent-regex without attachment ("transcribe this audio") → AUDIO
  • Why: ATIS provides the canonical TEXT case, arXiv the image-heavy
    queries, ActivityNet the videos. The test exercises the three
    classifier channels (attachments → blocks → intent regex).

Component description:
  `classify_modality(message, settings)` examines, in order:
    1. `additional_kwargs["attachments"][i].mime_type` → modality map.
    2. `message.content[i]["type"]` (image_url, input_audio, video, …).
    3. Intent regex (transcribe/voice → AUDIO, picture → IMAGE).
    4. Default `TEXT` if there is text, `UNKNOWN` otherwise.

  `make_modality_router_node(use_llm_fallback=False)` wraps the
  classifier in a LangGraph node whose output is:
    {"next": "<vision_agent|audio_agent|video_agent|fusion|text>",
     "metadata": {"mm": {"router": {modality, confidence, …}}}}

  `state["metadata"]["mm"]["force_modality"]` forces the decision (useful
  for tests and so the user can override the heuristic).

Usage:
    uv run python examples/multimodal/04_modality_router.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from langchain_core.messages import HumanMessage

from prismal.agents.multimodal import (
    Modality,
    classify_modality,
    make_modality_router_node,
)

## Dataset: 18 messages with expected modality

In [ ]:
CASES = [
    # === TEXT (ATIS) ===
    ("atis_t1", "Show me all flights from Boston to Denver", None, None, Modality.TEXT),
    ("atis_t2", "What's the cheapest fare to JFK", None, None, Modality.TEXT),
    ("atis_t3", "Cancel my reservation for tomorrow", None, None, Modality.TEXT),
    # === IMAGE via attachment ===
    ("img_a1", "What's in this photo?", "image/png", None, Modality.IMAGE),
    ("img_a2", "Describe the figure", "image/jpeg", None, Modality.IMAGE),
    # === IMAGE via content block ===
    ("img_b1", None, None, "image_url", Modality.IMAGE),
    # === IMAGE via intent regex (no attachment) ===
    ("img_r1", "Can you analyze the picture I uploaded yesterday?", None, None, Modality.IMAGE),
    ("img_r2", "Show me the screenshot from the meeting", None, None, Modality.IMAGE),
    # === AUDIO via attachment ===
    ("aud_a1", "Transcribe this", "audio/wav", None, Modality.AUDIO),
    ("aud_a2", "What did they say?", "audio/mpeg", None, Modality.AUDIO),
    # === AUDIO via content block ===
    ("aud_b1", "Listen", None, "input_audio", Modality.AUDIO),
    # === AUDIO via intent regex ===
    ("aud_r1", "Please transcribe the call from this morning", None, None, Modality.AUDIO),
    ("aud_r2", "I need to convert this voice memo to text", None, None, Modality.AUDIO),
    # === VIDEO via attachment ===
    ("vid_a1", "Summarize this clip", "video/mp4", None, Modality.VIDEO),
    # === VIDEO via intent regex ===
    ("vid_r1", "What happens in this video?", None, None, Modality.VIDEO),
    # === MIXED ===
    ("mix_1", "Compare these", "image/png,audio/wav", None, Modality.MIXED),
    ("mix_2", "Analyze", "video/mp4,image/jpeg", None, Modality.MIXED),
    # === UNKNOWN ===
    ("unk_1", "", None, None, Modality.UNKNOWN),
]


def build_message(text: str | None, attachments_mime: str | None, block_type: str | None):
    """Build a HumanMessage with attachments and/or content blocks."""
    kwargs: dict = {}
    if attachments_mime:
        kwargs["attachments"] = [
            {"mime_type": m.strip()} for m in attachments_mime.split(",")
        ]

    if block_type:
        content: list[dict] = []
        if text:
            content.append({"type": "text", "text": text})
        content.append({"type": block_type, "image_url": {"url": "data:..."}})
        return HumanMessage(content=content, additional_kwargs=kwargs)

    return HumanMessage(content=text or "", additional_kwargs=kwargs)


async def main() -> None:
    print("=" * 70)
    print("ModalityRouter · deterministic classification (no LLM)")
    print("=" * 70)

    # 1. Standard classify_modality
    print("\n" + "─" * 70)
    print("1) classify_modality(message) — 18 labeled cases")
    print("─" * 70)
    correct = 0
    for case_id, text, mime, block, expected in CASES:
        msg = build_message(text, mime, block)
        result = classify_modality(msg)
        ok = result.modality == expected
        correct += int(ok)
        mark = "✓" if ok else "✗"
        print(
            f"  {mark} {case_id:8} → {result.modality.value:8} "
            f"(conf={result.confidence:.2f})  expected={expected.value}"
        )
    print(f"\n  Accuracy: {correct}/{len(CASES)}  ({100 * correct / len(CASES):.1f}%)")

    # 2. LangGraph node (target routing)
    print("\n" + "─" * 70)
    print("2) make_modality_router_node — routing to LangGraph nodes")
    print("─" * 70)
    router = make_modality_router_node(use_llm_fallback=False)

    targets = {
        "vision_agent": [],
        "audio_agent": [],
        "video_agent": [],
        "fusion": [],
        "text": [],
    }
    for case_id, text, mime, block, _ in CASES:
        msg = build_message(text, mime, block)
        state = {"messages": [msg], "metadata": {}}
        result = await router(state)
        targets[result["next"]].append(case_id)

    for node, cases in targets.items():
        print(f"  → {node:14} · {len(cases):2} cases  {cases}")

    # 3. force_modality override
    print("\n" + "─" * 70)
    print("3) override with state.metadata.mm.force_modality")
    print("─" * 70)
    msg = HumanMessage(content="random text without intent")
    state = {
        "messages": [msg],
        "metadata": {"mm": {"force_modality": Modality.AUDIO.value}},
    }
    result = await router(state)
    print(f"\n  forced → {result['next']}  (router metadata: {result['metadata']['mm']['router']})")
    assert result["next"] == "audio_agent"

    print("\n" + "=" * 70)
    print("OK — ModalityRouter without LLM (use_llm_fallback=True optional)")
    print("=" * 70)


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()